# Documento de entrega - LAB1
Regresion con Spark MLlib (dataset en memoria)

## Informacion del estudiante
- Nombre y apellidos: Juan Manuel Vega
- Grupo: Big Data
- Fecha de entrega: 23/03/2026

## 1. Creacion del dataset
Creamos el dataset de viviendas en memoria con Spark usando createDataFrame.

In [1]:
import os
import sys
from pyspark.sql import SparkSession

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

try:
    spark.stop()
except Exception:
    pass

spark = (
    SparkSession.builder
    .appName('LAB1_Regresion_Spark')
    .master('local[1]')
    .config('spark.pyspark.python', sys.executable)
    .config('spark.pyspark.driver.python', sys.executable)
    .config('spark.executorEnv.PYSPARK_PYTHON', sys.executable)
    .getOrCreate()
)

spark

In [2]:
data = [
    (50, 2, 1, 120000),
    (60, 2, 1, 140000),
    (75, 3, 2, 180000),
    (80, 3, 2, 200000),
    (90, 3, 2, 220000),
    (100, 4, 2, 250000),
    (120, 4, 3, 300000),
    (150, 5, 3, 380000),
    (45, 1, 1, 100000),
    (110, 4, 2, 270000)
]

columns = ['size_m2', 'rooms', 'bathrooms', 'price']
dataset = spark.createDataFrame(data, columns)
dataset.show()

+-------+-----+---------+------+
|size_m2|rooms|bathrooms| price|
+-------+-----+---------+------+
|     50|    2|        1|120000|
|     60|    2|        1|140000|
|     75|    3|        2|180000|
|     80|    3|        2|200000|
|     90|    3|        2|220000|
|    100|    4|        2|250000|
|    120|    4|        3|300000|
|    150|    5|        3|380000|
|     45|    1|        1|100000|
|    110|    4|        2|270000|
+-------+-----+---------+------+



## 2. Preparacion de features
Creamos la columna features usando VectorAssembler.

In [3]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=['size_m2', 'rooms', 'bathrooms'],
    outputCol='features'
)

dataset_features = assembler.transform(dataset)
dataset_features.select('features', 'price').show(truncate=False)

+---------------+------+
|features       |price |
+---------------+------+
|[50.0,2.0,1.0] |120000|
|[60.0,2.0,1.0] |140000|
|[75.0,3.0,2.0] |180000|
|[80.0,3.0,2.0] |200000|
|[90.0,3.0,2.0] |220000|
|[100.0,4.0,2.0]|250000|
|[120.0,4.0,3.0]|300000|
|[150.0,5.0,3.0]|380000|
|[45.0,1.0,1.0] |100000|
|[110.0,4.0,2.0]|270000|
+---------------+------+



## 3. Division del dataset
Dividimos en entrenamiento y prueba con randomSplit.

In [4]:
train_data, test_data = dataset_features.randomSplit([0.8, 0.2], seed=42)
print(f'Filas entrenamiento: {train_data.count()}')
print(f'Filas prueba: {test_data.count()}')

Filas entrenamiento: 7
Filas prueba: 3


## 4. Entrenamiento del modelo
Entrenamos una regresion lineal con LinearRegression.

In [5]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol='features', labelCol='price')
model = lr.fit(train_data)

print('Coeficientes:', model.coefficients)
print('Intercepto:', model.intercept)

Coeficientes: [2477.272727272717,2931.8181818181024,2363.6363636370643]
Intercepto: -15522.727272727425


## 5. Predicciones
Generamos predicciones con model.transform().

In [6]:
predictions = model.transform(test_data)
predictions.select('features', 'price', 'prediction').show(truncate=False)

+---------------+------+------------------+
|features       |price |prediction        |
+---------------+------+------------------+
|[60.0,2.0,1.0] |140000|141340.90909090883|
|[100.0,4.0,2.0]|250000|248659.09090909082|
|[120.0,4.0,3.0]|300000|300568.18181818223|
+---------------+------+------------------+



## 6. Evaluacion del modelo
Calculamos la metrica RMSE con RegressionEvaluator.

In [7]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol='price',
    predictionCol='prediction',
    metricName='rmse'
)

rmse = evaluator.evaluate(predictions)
print(f'RMSE del modelo: {rmse:.2f}')

RMSE del modelo: 1142.94


## Preguntas de reflexion
1. Que variables se utilizan como features?
Se usan size_m2, rooms y bathrooms.

2. Cual es la variable objetivo del modelo?
La variable objetivo es price.

3. Que funcion cumple VectorAssembler?
Combina columnas numericas en un vector de entrada para MLlib.

4. Por que es necesario dividir el dataset en entrenamiento y prueba?
Para evaluar en datos no vistos y evitar sesgo.

5. Que representa la metrica RMSE?
El error medio de prediccion en las mismas unidades de la variable objetivo.